# Portfolio optimization: maximum Sharpe ratio

Reusable each FinSimCo quarter. Drop the new price file in `data/` and run `./run_quarter.sh Q2` (see `README.md`).

Method:
- Input: daily prices, one column per ticker, first column `Date`
- Missing prices filled with the previous row's value (forward fill)
- Simple daily returns, annualized with 252 trading days
- Sections 2 to 4: long-only, fully invested (weights between 0 and 1, sum to 1)
- Section 5: leverage up to `LEVERAGE` gross exposure (borrowing at the margin rate, or long/short with short rebate and borrow fee)
- Section 6: weight changes vs. the previous quarter, if it was run

All inputs are in the parameters cell below.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# ---- Parameters: change these each quarter (or pass QUARTER / DATA_FILE as env vars) ----
QUARTER = os.environ.get("QUARTER", "Q1")
DATA_FILE = Path(os.environ.get("DATA_FILE", f"data/Historical Data {QUARTER}.xlsx"))

RF = 0.04             # annual risk-free rate
LEVERAGE = 2.0        # max gross exposure / capital
MARGIN_RATE = RF      # cost of borrowing cash (A and B)
SHORT_REBATE = RF     # interest earned on short sale proceeds
BORROW_FEE = 0.005    # annual fee to borrow shares (placeholder, use broker rate)
MAX_MISSING = 0.30    # drop a ticker if more than this share of its prices is missing

DAYS = 252
N_SIM = 50_000
SEED = 42
# -----------------------------------------------------------------------------------------

OUT = Path("outputs") / QUARTER
OUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print(f"Quarter: {QUARTER} | data: {DATA_FILE} | outputs: {OUT}/")

## 1. Data and cleaning


In [ ]:
assert DATA_FILE.exists(), f"Data file not found: {DATA_FILE}"
raw = pd.read_excel(DATA_FILE)
date_col = raw.columns[0]
raw[date_col] = pd.to_datetime(raw[date_col])
prices = raw.set_index(date_col).sort_index().apply(pd.to_numeric, errors="coerce")
prices = prices[~prices.index.duplicated(keep="last")].dropna(how="all", axis=1)

missing = prices.isna().mean()
dropped = missing[missing > MAX_MISSING]
if len(dropped):
    print("Dropped tickers (too much missing data):", dropped.round(2).to_dict())
prices = prices.drop(columns=dropped.index)

print("Missing before fill:", int(prices.isna().sum().sum()))
prices = prices.ffill().dropna()  # leading rows with no prior price are dropped
print("Missing after fill:", int(prices.isna().sum().sum()))
print(f"{prices.shape[1]} tickers, {len(prices)} days, {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}")
assert len(prices) > 60, "Too few rows after cleaning"
prices.head()

In [ ]:
returns = prices.pct_change().dropna()
mu = returns.mean() * DAYS
cov = returns.cov() * DAYS
vol = np.sqrt(np.diag(cov))
tickers = list(prices.columns)
n = len(tickers)

stats = pd.DataFrame(
    {"Annual return": mu, "Annual volatility": vol, "Sharpe": (mu - RF) / vol}
).sort_values("Sharpe", ascending=False)
stats.style.format("{:.2%}", subset=["Annual return", "Annual volatility"]).format(
    "{:.2f}", subset=["Sharpe"]
)

## 2. Monte Carlo simulation of random portfolios


In [ ]:
def perf(w):
    r = w @ mu.values
    s = np.sqrt(w @ cov.values @ w)
    return r, s, (r - RF) / s


W = np.random.dirichlet(np.ones(n), N_SIM)
sim_ret = W @ mu.values
sim_vol = np.sqrt(np.einsum("ij,jk,ik->i", W, cov.values, W))
sim_sharpe = (sim_ret - RF) / sim_vol
best_sim = sim_sharpe.argmax()
print(
    f"Best simulated portfolio: return {sim_ret[best_sim]:.2%}, vol {sim_vol[best_sim]:.2%}, Sharpe {sim_sharpe[best_sim]:.2f}"
)

## 3. Optimization (SLSQP)


In [ ]:
bounds = [(0, 1)] * n
cons = ({"type": "eq", "fun": lambda w: w.sum() - 1},)
w0 = np.ones(n) / n

opt_sharpe = minimize(
    lambda w: -perf(w)[2], w0, method="SLSQP", bounds=bounds, constraints=cons
)
opt_minvar = minimize(
    lambda w: perf(w)[1], w0, method="SLSQP", bounds=bounds, constraints=cons
)
assert opt_sharpe.success and opt_minvar.success

w_sharpe, w_minvar = opt_sharpe.x, opt_minvar.x
summary = pd.DataFrame(
    {
        name: perf(w)
        for name, w in [
            ("Max Sharpe", w_sharpe),
            ("Min variance", w_minvar),
            ("Equal weight", w0),
        ]
    },
    index=["Return", "Volatility", "Sharpe"],
).T
summary.style.format("{:.2%}", subset=["Return", "Volatility"]).format(
    "{:.2f}", subset=["Sharpe"]
)

In [ ]:
weights = pd.DataFrame(
    {"Max Sharpe": w_sharpe, "Min variance": w_minvar}, index=tickers
)
weights = weights[(weights > 1e-4).any(axis=1)].sort_values(
    "Max Sharpe", ascending=False
)
weights.style.format("{:.2%}")

## 4. Efficient frontier and capital market line


In [ ]:
targets = np.linspace(perf(w_minvar)[0], mu.max(), 60)
frontier = []
for t in targets:
    c = cons + ({"type": "eq", "fun": lambda w, t=t: w @ mu.values - t},)
    r = minimize(lambda w: perf(w)[1], w0, method="SLSQP", bounds=bounds, constraints=c)
    frontier.append(r.fun if r.success else np.nan)

r_s, s_s, sh_s = perf(w_sharpe)
r_m, s_m, _ = perf(w_minvar)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(sim_vol, sim_ret, c=sim_sharpe, cmap="viridis", s=3, alpha=0.5)
plt.colorbar(sc, label="Sharpe ratio")
ax.plot(frontier, targets, color="black", lw=2, label="Efficient frontier")
x = np.linspace(0, max(sim_vol.max(), s_s) * 1.1, 50)
ax.plot(x, RF + sh_s * x, "--", color="gray", label="Capital market line")
ax.scatter(vol, mu, marker="x", color="gray", s=25)
for t, v, m in zip(tickers, vol, mu):
    ax.annotate(t, (v, m), fontsize=7, color="gray")
ax.scatter(s_s, r_s, marker="*", color="red", s=300, label=f"Max Sharpe ({sh_s:.2f})")
ax.scatter(s_m, r_m, marker="*", color="blue", s=300, label="Min variance")
ax.set_xlabel("Annual volatility")
ax.set_ylabel("Annual return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"{QUARTER}: simulated portfolios, efficient frontier and CML (rf = {RF:.1%})")
ax.legend(loc="lower right")
plt.tight_layout()
fig.savefig(OUT / "frontier.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ws = weights["Max Sharpe"][weights["Max Sharpe"] > 1e-4]
ax.bar(ws.index, ws.values, color="tab:red")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title(f"{QUARTER}: max Sharpe portfolio weights")
plt.tight_layout()
fig.savefig(OUT / "weights_max_sharpe.png", dpi=150)
plt.show()

## 5. Leverage (hedge fund)

Two ways to use leverage, both capped at a gross exposure of `LEVERAGE` times capital:

- **A. Borrow at the risk-free rate and lever the max Sharpe portfolio.** Return and volatility scale up together along the CML, so the **Sharpe ratio does not change**.
- **B. Long/short.** Net exposure 100%, gross exposure up to `LEVERAGE`. This widens the opportunity set, so the **Sharpe ratio can rise**.

Financing for B, per year:

- Longs above 100% of capital are bought on margin at `MARGIN_RATE`.
- Short sale proceeds stay with the broker as collateral and earn `SHORT_REBATE`.
- Each short pays `BORROW_FEE` on its value to the stock lender.

So B's return = expected stock returns + short × `SHORT_REBATE` − (long − 100%) × `MARGIN_RATE` − short × `BORROW_FEE`.
The optimizer includes these costs, so a higher fee leads to fewer shorts.

Not modeled: margin calls, hard-to-borrow stocks with higher fees, a spread on margin above the risk-free rate.


In [ ]:
# A. Levered tangency portfolio: L in max Sharpe, (1 - L) in cash (negative = borrowing)
lev_ret = MARGIN_RATE + LEVERAGE * (r_s - MARGIN_RATE)
lev_vol = LEVERAGE * s_s
w_lev = LEVERAGE * w_sharpe


# B. Long/short max Sharpe: w = long - short, sum(w) = 1, sum(long + short) <= LEVERAGE
def split(x):
    return x[:n] - x[n:]


def carry(x, rebate, margin, fee):
    long_, short = x[:n].sum(), x[n:].sum()
    return short * rebate - max(long_ - 1, 0) * margin - short * fee


def perf_ls(x, rebate=SHORT_REBATE, margin=MARGIN_RATE, fee=BORROW_FEE):
    r, s, _ = perf(split(x))
    r += carry(x, rebate, margin, fee)
    return r, s, (r - RF) / s


def opt_long_short(**kw):
    cons = (
        {"type": "eq", "fun": lambda x: split(x).sum() - 1},
        {"type": "ineq", "fun": lambda x: LEVERAGE - x.sum()},
    )
    x0 = np.concatenate([w0, np.zeros(n)])
    res = minimize(
        lambda x: -perf_ls(x, **kw)[2],
        x0,
        method="SLSQP",
        bounds=[(0, LEVERAGE)] * (2 * n),
        constraints=cons,
        options={"maxiter": 1000},
    )
    assert res.success, res.message
    x = res.x.copy()
    x[x < 1e-6] = 0
    return x


x_ls0 = opt_long_short(
    rebate=0, margin=0, fee=0
)  # previous version: no financing at all
x_ls = opt_long_short()  # with rebate, margin and borrow fee
w_ls0, w_ls = split(x_ls0), split(x_ls)


def row(r, s, w, borrow=0.0):
    return {
        "Return": r,
        "Volatility": s,
        "Sharpe": (r - RF) / s,
        "Long": w[w > 0].sum(),
        "Short": -w[w < 0].sum(),
        "Gross": np.abs(w).sum(),
        "Borrowed": borrow,
    }


lev_summary = pd.DataFrame(
    {
        "Max Sharpe (no leverage)": row(r_s, s_s, w_sharpe),
        f"A. Levered max Sharpe ({LEVERAGE:.0f}x)": row(
            lev_ret, lev_vol, w_lev, borrow=LEVERAGE - 1
        ),
        "B. Long/short, no financing": row(
            *perf_ls(x_ls0, 0, 0, 0)[:2],
            w_ls0,
            borrow=max(w_ls0[w_ls0 > 0].sum() - 1, 0),
        ),
        f"B. Long/short, rebate + {BORROW_FEE:.1%} fee": row(
            *perf_ls(x_ls)[:2], w_ls, borrow=max(w_ls[w_ls > 0].sum() - 1, 0)
        ),
    }
).T
lev_summary.style.format(
    "{:.2%}", subset=["Return", "Volatility", "Long", "Short", "Gross", "Borrowed"]
).format("{:.2f}", subset=["Sharpe"])

In [ ]:
lev_weights = pd.DataFrame(
    {
        "Max Sharpe": w_sharpe,
        f"A. Levered {LEVERAGE:.0f}x": w_lev,
        "B. No financing": w_ls0,
        "B. Rebate + fee": w_ls,
    },
    index=tickers,
)
lev_weights = lev_weights[(lev_weights.abs() > 1e-4).any(axis=1)].sort_values(
    "B. Rebate + fee", ascending=False
)
lev_weights.style.format("{:.2%}")

In [ ]:
r_ls, s_ls, sh_ls = perf_ls(x_ls)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    sim_vol, sim_ret, c="lightgray", s=3, alpha=0.5, label="Random long-only portfolios"
)
ax.plot(frontier, targets, color="black", lw=2, label="Long-only efficient frontier")
x = np.linspace(0, max(lev_vol, s_ls) * 1.2, 50)
ax.plot(
    x, RF + sh_s * x, "--", color="tab:red", label=f"CML long-only (Sharpe {sh_s:.2f})"
)
ax.plot(
    x,
    RF + sh_ls * x,
    "--",
    color="tab:purple",
    label=f"CML long/short (Sharpe {sh_ls:.2f})",
)
ax.scatter(s_s, r_s, marker="*", color="red", s=300, label="Max Sharpe, no leverage")
ax.scatter(
    lev_vol,
    lev_ret,
    marker="*",
    color="darkred",
    s=300,
    label=f"A. Levered {LEVERAGE:.0f}x",
)
ax.scatter(
    s_ls,
    r_ls,
    marker="*",
    color="purple",
    s=300,
    label=f"B. Long/short {LEVERAGE:.0f}x gross, rebate + fee",
)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Annual volatility")
ax.set_ylabel("Annual return")
ax.set_title(f"{QUARTER}: leverage, moving along the CML vs. long/short")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "leverage.png", dpi=150)
plt.show()

## 6. Changes vs. previous quarter

Trades needed to move from last quarter's weights to this quarter's. Skipped if the previous quarter has not been run.

In [ ]:
prev_q = f"Q{int(QUARTER[1:]) - 1}" if QUARTER[1:].isdigit() and int(QUARTER[1:]) > 1 else None
prev_file = Path("outputs") / str(prev_q) / f"resultados_sharpe_{prev_q}.xlsx"
if prev_q and prev_file.exists():
    prev_w = pd.read_excel(prev_file, sheet_name="Leverage weights", index_col=0)
    cur_w = pd.DataFrame({"Max Sharpe": w_sharpe, "A. Levered": w_lev, "B. Rebate + fee": w_ls}, index=tickers)
    cols = ["Max Sharpe", "A. Levered", "B. Rebate + fee"]
    change = cur_w[cols].sub(prev_w.reindex(columns=cols), fill_value=0).fillna(0)
    change = change[(change.abs() > 1e-4).any(axis=1)].sort_values("B. Rebate + fee")
    print(f"Weight change {prev_q} -> {QUARTER} (positive = buy, negative = sell)")
    display(change.style.format("{:+.2%}"))
else:
    change = None
    print(f"No previous quarter results found ({prev_file}), skipping comparison")

## 7. Export

In [ ]:
xlsx = OUT / f"resultados_sharpe_{QUARTER}.xlsx"
params = pd.Series({"Quarter": QUARTER, "Data file": str(DATA_FILE), "Start": prices.index.min().date(),
                    "End": prices.index.max().date(), "Days": len(prices), "Tickers": len(tickers),
                    "Dropped tickers": ", ".join(dropped.index) or "none", "RF": RF, "Leverage": LEVERAGE,
                    "Margin rate": MARGIN_RATE, "Short rebate": SHORT_REBATE, "Borrow fee": BORROW_FEE},
                   name="Value")
with pd.ExcelWriter(xlsx) as xw:
    params.to_excel(xw, sheet_name="Parameters")
    summary.to_excel(xw, sheet_name="Summary")
    pd.DataFrame({"Max Sharpe": w_sharpe, "Min variance": w_minvar}, index=tickers).to_excel(xw, sheet_name="Weights")
    lev_summary.to_excel(xw, sheet_name="Leverage summary")
    pd.DataFrame({"Max Sharpe": w_sharpe, "A. Levered": w_lev, "B. No financing": w_ls0, "B. Rebate + fee": w_ls},
                 index=tickers).to_excel(xw, sheet_name="Leverage weights")
    if change is not None:
        change.to_excel(xw, sheet_name="Change vs prev quarter")
    stats.to_excel(xw, sheet_name="Asset stats")
    cov.to_excel(xw, sheet_name="Covariance")
print(f"Saved {xlsx} and charts in {OUT}/")